# Dependencies


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from datetime import datetime


PATH = "../data/raw/untouched_raw_original.csv"
OUTPUT = "../reports/"

# Insights


In [2]:
df = pd.read_csv(PATH)
print(f"Loaded {len(df):,} rows x {len(df.columns)} columns")

Loaded 14,340 rows x 28 columns


In [3]:
housing_schema = []
for col in df.columns:
    dtype = df[col].dtype
    n_unique = df[col].nunique()
    n_missing = df[col].isna().sum()
    pct_missing = 100 * n_missing / len(df)
    sample_df = df[col].dropna().head(3).to_list()

    housing_schema.append(
        {
            "column": col,
            "dtype": dtype,
            "unique_count": n_unique,
            "missing_count": n_missing,
            "pct_missing": f"{pct_missing:.2f}%",
            "sample_values": str(sample_df)[:60],
        }
    )

schema_df = pd.DataFrame(housing_schema)
# print(schema_df.to_string(index=False))
schema_df.to_csv(f"{OUTPUT}data_schema.csv", index=False)

In [4]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_pct = (100 * missing_summary / len(df)).round(2)
missing_cols = df.columns[
    df.isna().any()
].tolist() 
missing_cols_str = ", ".join(missing_cols) if missing_cols else "None"
missing_report = pd.DataFrame(
    {"missing_count": missing_summary, "missing_pct": missing_pct}
).query("missing_count > 0")


In [5]:
prices = df["price"].dropna()

# Raw price stats
price_stats = {
    "count": len(prices),
    "mean": prices.mean(),
    "median": prices.median(),
    "std": prices.std(),
    "min": prices.min(),
    "max": prices.max(),
    "q25": prices.quantile(0.25),
    "q75": prices.quantile(0.75),
    "iqr": prices.quantile(0.75) - prices.quantile(0.25),
}

print("\nRaw Price Statistics:")
for k, v in price_stats.items():
    print(f"  {k:10s}: {v:,.2f}" if isinstance(v, float) else f"  {k:10s}: {v:,}")

# Log price statistics
log_prices = np.log(prices[prices > 0])
log_stats = {
    "count": len(log_prices),
    "mean": log_prices.mean(),
    "median": log_prices.median(),
    "std": log_prices.std(),
}

print("\nLog Price Statistics:")
for k, v in log_stats.items():
    print(f"  {k:10s}: {v:,.4f}" if isinstance(v, float) else f"  {k:10s}: {v:,}")



Raw Price Statistics:
  count     : 14,340
  mean      : 162,621.69
  median    : 5,000.00
  std       : 14,079,428.76
  min       : 175.00
  max       : 1,655,500,000.00
  q25       : 2,300.00
  q75       : 13,500.00
  iqr       : 11,200.00

Log Price Statistics:
  count     : 14,340
  mean      : 8.6219
  median    : 8.5172
  std       : 1.2417


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Raw price histogram
axes[0, 0].hist(prices, bins=50, edgecolor="black", alpha=0.7)
axes[0, 0].set_xlabel("Price")
axes[0, 0].set_ylabel("Frequency")
axes[0, 0].set_title("Raw Price Distribution")

# Raw price boxplot
axes[0, 1].boxplot(prices, vert=True)
axes[0, 1].set_ylabel("Price")
axes[0, 1].set_title("Raw Price Boxplot")

# Log price histogram
axes[1, 0].hist(log_prices, bins=50, edgecolor="black", alpha=0.7, color="green")
axes[1, 0].set_xlabel("Log(Price)")
axes[1, 0].set_ylabel("Frequency")
axes[1, 0].set_title("Log Price Distribution")

# Q-Q plot for log prices

stats.probplot(log_prices, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Q-Q Plot (Log Price)")

plt.tight_layout()
plt.savefig(f"{OUTPUT}price_distributions.png", dpi=150)
plt.close()
print("✓ Saved price distribution visualizations")


✓ Saved price distribution visualizations


In [7]:
n_exact_dupes = df.duplicated().sum()
pct_exact_dupes = 100 * n_exact_dupes / len(df)
print(f"Exact duplicate rows: {n_exact_dupes:,} ({pct_exact_dupes:.2f}%)")

if "url" in df.columns or "id" in df.columns:
    id_col = "url" if "url" in df.columns else "id"
    n_id_dupes = df[id_col].duplicated().sum()
    pct_id_dupes = 100 * n_id_dupes / len(df)
    print(f"Duplicate {id_col}s: {n_id_dupes:,} ({pct_id_dupes:.2f}%)")

    dupe_counts = df[id_col].value_counts()
    print(f"\nListings appearing multiple times: {(dupe_counts > 1).sum():,}")
    print(f"Max appearances: {dupe_counts.max()}")

    if (dupe_counts > 1).sum() > 0:
        plt.figure(figsize=(10, 6))
        dupe_counts[dupe_counts > 1].hist(bins=30, edgecolor="black")
        plt.xlabel("Number of Appearances")
        plt.ylabel("Number of Listings")
        plt.title("Distribution of Duplicate Listings")
        plt.tight_layout()
        plt.savefig(f"{OUTPUT}duplicate_listings.png", dpi=150)
        plt.close()


Exact duplicate rows: 0 (0.00%)
Duplicate urls: 0 (0.00%)

Listings appearing multiple times: 0
Max appearances: 1


In [9]:
# Listings per neighborhood
nbhd_counts = df["locality"].value_counts()
print(f"\nTotal neighborhoods: {len(nbhd_counts)}")
print(f"Neighborhoods with <10 listings: {(nbhd_counts < 10).sum()}")
print(f"Neighborhoods with <50 listings: {(nbhd_counts < 50).sum()}")
print(f"Neighborhoods with <100 listings: {(nbhd_counts < 100).sum()}")

print("\nTop 10 neighborhoods by listing count:")
print(nbhd_counts.head(10))

print("\nBottom 10 neighborhoods by listing count:")
print(nbhd_counts.tail(10))

# Save full neighborhood summary
nbhd_summary = pd.DataFrame(
    {
        "neighborhood": nbhd_counts.index,
        "listing_count": nbhd_counts.values,
        "pct_of_total": 100 * nbhd_counts.values / len(df),
    }
)
nbhd_summary.to_csv(f"{OUTPUT}neighborhood_summary.csv", index=False)
print("✓ Saved locality statistics")


# Monthly analysis if date available
# if "date" in df.columns:
#     df["month"] = pd.to_datetime(df["date"]).dt.to_period("M")
#     monthly_nbhd = (
#         df.groupby([neighborhood_col, "month"]).size().reset_index(name="count")
#     )

#     # Neighborhoods per month
#     nbhd_per_month = monthly_nbhd.groupby("month")[neighborhood_col].nunique()
#     print(f"\nAverage neighborhoods active per month: {nbhd_per_month.mean():.1f}")

#     # Listings per neighborhood per month
#     avg_listings_per_nbhd_month = monthly_nbhd["count"].mean()
#     print(
#         f"Average listings per neighborhood per month: {avg_listings_per_nbhd_month:.1f}"
#     )

#     monthly_nbhd.to_csv(f"{OUTPUT_DIR}neighborhood_monthly.csv", index=False)



Total neighborhoods: 84
Neighborhoods with <10 listings: 20
Neighborhoods with <50 listings: 41
Neighborhoods with <100 listings: 54

Top 10 neighborhoods by listing count:
locality
East Legon                  1662
Teshie                      1167
Spintex                     1051
Adenta                       841
Tema Metropolitan            753
Accra Metropolitan           621
Ashaley Botwe                576
Airport Residential Area     560
Adjiriganor                  535
Weija                        502
Name: count, dtype: int64

Bottom 10 neighborhoods by listing count:
locality
Nima              3
Bubuashie         3
Abofu             3
South Shiashie    3
Okponglo          3
Circle            3
Adabraka          2
Korle Gonno       2
Banana Inn        1
South La          1
Name: count, dtype: int64
✓ Saved locality statistics


In [ ]:
summary = f"""
# Phase 1: Data Reality Check Summary
**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

## Dataset Overview
- **Total rows:** {len(df):,}
- **Total columns:** {len(df.columns)}
- **Date range:** {df["date"].min() if "date" in df.columns else "N/A"} to {df["date"].max() if "date" in df.columns else "N/A"}

## Data Quality Issues
- **Number of columns with missing data:** {len(missing_report) if len(missing_report) > 0 else 0}
- **Missing data columns:** {missing_cols_str}
- **Highest missing %:** {missing_report["missing_pct"].max() if len(missing_report) > 0 else 0}%
- **Exact duplicate rows:** {n_exact_dupes:,} ({pct_exact_dupes:.2f}%)

## Price Distribution
- **Mean price:** ${price_stats["mean"]:,.2f}
- **Median price:** ${price_stats["median"]:,.2f}
- **Price range:** ${price_stats["min"]:,.2f} - ${price_stats["max"]:,.2f}
- **Log price std:** {log_stats["std"]:.4f}

## Spatial Coverage
- **Total neighborhoods:** {len(nbhd_counts)}
- **Data-poor neighborhoods (<50 listings):** {(nbhd_counts < 50).sum()}
- **Data-poor neighborhoods (<100 listings):** {(nbhd_counts < 100).sum()}

## Key Questions to Answer

### 1. Which features are unusable?
- Review `schema_inspection.csv` for high missing % columns
- Review `missing_values.png` visualization
- **Decision threshold:** Consider dropping features with >50% missing

### 2. Which neighborhoods are data-poor?
- Review `neighborhood_summary.csv`
- Review `neighborhood_analysis.png`
- **Decision threshold:** Flag neighborhoods with <50-100 listings for shrinkage

### 3. What is the minimum sample size for modeling?
- Based on neighborhood analysis: ____ listings minimum
- Based on temporal coverage: ____ months minimum
- **Recommendation:** Set explicit thresholds before Phase 2

### 4. Is shrinkage/regularization justified?
- **Yes** if many data-poor neighborhoods exist
- **Yes** if high feature-to-sample ratio in small neighborhoods


## Files Generated
- `data_schema.csv`
- `price_distributions.png`
- `neighborhood_summary.csv`
- `neighborhood_monthly.csv` (if date available)
- `duplicate_listings.png` (if duplicates found)
"""

# Save summary
with open(f"{OUTPUT}PHASE1_SUMMARY.md", "w") as f:
    f.write(summary)
